In [14]:
# ==================================================
# 재학습용 보강 데이터셋 준비
# 목적: user_id는 탐지됐지만 user_character가 미탐지된 이미지들을 별도 폴더로 복사
# ==================================================

from pathlib import Path
import shutil
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

TEST_IMAGES = PROJECT_ROOT / "data" / "test" / "images"

RETRAIN_IMAGES = PROJECT_ROOT / "data" / "retrain" / "images"
RETRAIN_LABELS = PROJECT_ROOT / "data" / "retrain" / "labels"

RETRAIN_IMAGES.mkdir(parents=True, exist_ok=True)
RETRAIN_LABELS.mkdir(parents=True, exist_ok=True)

print("프로젝트 루트:", PROJECT_ROOT)
print("보강 이미지 폴더:", RETRAIN_IMAGES)
print("보강 라벨 폴더:", RETRAIN_LABELS)

프로젝트 루트: c:\Users\금정산2-PC16\Desktop\NS-Project
보강 이미지 폴더: c:\Users\금정산2-PC16\Desktop\NS-Project\data\retrain\images
보강 라벨 폴더: c:\Users\금정산2-PC16\Desktop\NS-Project\data\retrain\labels


In [15]:
# ==================================================
# user_character 미탐지 이미지 복사
# 목적: 재학습용 이미지 폴더 생성
# ==================================================

from pathlib import Path
import shutil

PROJECT_ROOT = Path.cwd().parent

RETRAIN_IMAGES = PROJECT_ROOT / "data" / "retrain" / "images"
RETRAIN_IMAGES.mkdir(parents=True, exist_ok=True)

target_files = [
    "ScreenShot2024_0304_141339663.jpg",
    "ScreenShot2024_0308_213117869.jpg",
    "ScreenShot2024_0309_202918770.jpg",
    "ScreenShot2024_0310_194737263.jpg",
    "ScreenShot2024_0311_164506563.jpg",
    "ScreenShot2024_0311_181930117.jpg",
    "ScreenShot2024_0318_011013069.jpg",
    "ScreenShot2024_0318_011800183.jpg",
    "ScreenShot2024_0318_213721758.jpg",
    "ScreenShot2024_0319_002449059.jpg",
    "ScreenShot2024_0319_003443559.jpg",
    "ScreenShot2024_0319_015320897.jpg",
    "ScreenShot2024_0319_185347294.jpg",
    "ScreenShot2024_0320_033418937.jpg",
    "ScreenShot2024_0320_141305854.jpg",
    "ScreenShot2024_0321_210051007.jpg",
    "ScreenShot2024_0713_030125785.jpg",
    "ScreenShot2025_0204_011534257.jpg",
    "ScreenShot2025_0218_012034214.jpg",
    "ScreenShot2025_0405_000547458.jpg",
    "ScreenShot2025_0501_010757510.jpg",
    "ScreenShot2025_1011_111835902.jpg",
    "ScreenShot2025_1119_142153717.jpg"
]

for filename in target_files:

    src = TEST_IMAGES / filename
    dst = RETRAIN_IMAGES / filename

    if src.exists():
        shutil.copy2(src, dst)

print("복사 완료")
print("이미지 수:", len(list(RETRAIN_IMAGES.glob("*.jpg"))))

복사 완료
이미지 수: 23


In [16]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

BEST_MODEL = (
    PROJECT_ROOT
    / "runs"
    / "detect"
    / "ns_yolov8s_960"
    / "weights"
    / "best.pt"
)

print(BEST_MODEL)
print(BEST_MODEL.exists())

c:\Users\금정산2-PC16\Desktop\NS-Project\runs\detect\ns_yolov8s_960\weights\best.pt
True


In [17]:
# ==================================================
# 학습 완료된 YOLOv8s 모델 로드
# 목적: 재학습용 이미지 자동 라벨링에 사용할 best 모델 불러오기
#
# 현재 노트북은 notebooks 폴더 안에서 실행되므로
# 프로젝트 루트를 기준으로 모델 경로를 생성한다.
# ==================================================

from ultralytics import YOLO

# 최종 학습된 YOLOv8s-960 모델 불러오기
model = YOLO(str(BEST_MODEL))

print("모델 로드 완료")

모델 로드 완료


In [18]:
# ==================================================
# 재학습 후보 이미지 자동 라벨링
# 목적: 기존 best 모델을 이용해 user_character,
#       user_id 박스를 자동 생성
#
# 결과:
# data/retrain/images/*.jpg
# ↓
# runs/detect/predict*/labels/*.txt
# ==================================================

results = model.predict(
    source=str(RETRAIN_IMAGES),
    conf=0.25,
    save_txt=True,
    save_conf=True,
    imgsz=960,
    device=0
)

print("자동 라벨링 완료")


image 1/23 c:\Users\2-PC16\Desktop\NS-Project\data\retrain\images\ScreenShot2024_0304_141339663.jpg: 512x960 1 user_id, 8.9ms
image 2/23 c:\Users\2-PC16\Desktop\NS-Project\data\retrain\images\ScreenShot2024_0308_213117869.jpg: 512x960 1 user_id, 8.1ms
image 3/23 c:\Users\2-PC16\Desktop\NS-Project\data\retrain\images\ScreenShot2024_0309_202918770.jpg: 512x960 1 user_character, 3 user_ids, 8.3ms
image 4/23 c:\Users\2-PC16\Desktop\NS-Project\data\retrain\images\ScreenShot2024_0310_194737263.jpg: 512x960 1 user_id, 8.9ms
image 5/23 c:\Users\2-PC16\Desktop\NS-Project\data\retrain\images\ScreenShot2024_0311_164506563.jpg: 512x960 1 user_character, 1 user_id, 10.3ms
image 6/23 c:\Users\2-PC16\Desktop\NS-Project\data\retrain\images\ScreenShot2024_0311_181930117.jpg: 512x960 1 user_id, 9.6ms
image 7/23 c:\Users\2-PC16\Desktop\NS-Project\data\retrain\images\ScreenShot2024_0318_011013069.jpg: 448x960 1 user_id, 8.3ms
image 8/23 c:\Users\2-PC16\Desktop\NS-Project\data\retrain\images\ScreenShot202

In [19]:
# ==================================================
# 가장 최근 자동 라벨링 결과 폴더 찾기
# 목적: predict-5처럼 번호가 바뀌어도 최신 labels 폴더를 자동으로 사용
# ==================================================

predict_dirs = sorted(
    (PROJECT_ROOT / "runs" / "detect").glob("predict*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

LATEST_PREDICT_DIR = predict_dirs[0]
LABEL_DIR = LATEST_PREDICT_DIR / "labels"

print("최근 predict 폴더:", LATEST_PREDICT_DIR)
print("라벨 폴더:", LABEL_DIR)
print("라벨 폴더 존재 여부:", LABEL_DIR.exists())

최근 predict 폴더: c:\Users\금정산2-PC16\Desktop\NS-Project\runs\detect\predict-6
라벨 폴더: c:\Users\금정산2-PC16\Desktop\NS-Project\runs\detect\predict-6\labels
라벨 폴더 존재 여부: True


In [20]:
# ==================================================
# 이미지별 user_character 탐지 여부 확인
# ==================================================

character_detected = 0
character_missing = []

for txt_file in LABEL_DIR.glob("*.txt"):

    has_character = False

    with open(txt_file, "r", encoding="utf-8") as f:

        for line in f:

            cls_id = int(line.split()[0])

            if cls_id == 0:
                has_character = True

    if has_character:
        character_detected += 1
    else:
        character_missing.append(txt_file.name)

print("character 탐지 성공:", character_detected)
print("character 탐지 실패:", len(character_missing))

character 탐지 성공: 8
character 탐지 실패: 15


In [21]:
# ==================================================
# user_character 자동 탐지 실패 파일 확인
# 목적: 수동 보강이 필요한 이미지 목록 확인
# ==================================================

character_missing

['ScreenShot2024_0304_141339663.txt',
 'ScreenShot2024_0308_213117869.txt',
 'ScreenShot2024_0310_194737263.txt',
 'ScreenShot2024_0311_181930117.txt',
 'ScreenShot2024_0318_011013069.txt',
 'ScreenShot2024_0318_011800183.txt',
 'ScreenShot2024_0318_213721758.txt',
 'ScreenShot2024_0319_002449059.txt',
 'ScreenShot2024_0319_003443559.txt',
 'ScreenShot2024_0319_185347294.txt',
 'ScreenShot2024_0320_033418937.txt',
 'ScreenShot2024_0320_141305854.txt',
 'ScreenShot2024_0713_030125785.txt',
 'ScreenShot2025_0204_011534257.txt',
 'ScreenShot2025_0405_000547458.txt']

In [22]:
# ==================================================
# character 미탐지 이미지 복사
# 목적: 수동 라벨 보강이 필요한 이미지 별도 관리
# ==================================================

import shutil

MISSING_IMAGES = PROJECT_ROOT / "data" / "retrain" / "missing_character_images"
MISSING_IMAGES.mkdir(parents=True, exist_ok=True)

for txt_name in character_missing:
    img_name = txt_name.replace(".txt", ".jpg")

    src = RETRAIN_IMAGES / img_name
    dst = MISSING_IMAGES / img_name

    if src.exists():
        shutil.copy2(src, dst)

print("복사 완료:", len(list(MISSING_IMAGES.glob('*.jpg'))))
print("저장 위치:", MISSING_IMAGES)

복사 완료: 15
저장 위치: c:\Users\금정산2-PC16\Desktop\NS-Project\data\retrain\missing_character_images


In [23]:
# ==================================================
# 자동 생성 라벨 복사
# 목적: predict-5/labels에 생성된 txt를 retrain/labels로 이동
# ==================================================

import shutil

RETRAIN_LABELS = PROJECT_ROOT / "data" / "retrain" / "labels"
RETRAIN_LABELS.mkdir(parents=True, exist_ok=True)

for txt_file in LABEL_DIR.glob("*.txt"):
    dst = RETRAIN_LABELS / txt_file.name
    shutil.copy2(txt_file, dst)

print("라벨 복사 완료:", len(list(RETRAIN_LABELS.glob("*.txt"))))
print("저장 위치:", RETRAIN_LABELS)

라벨 복사 완료: 23
저장 위치: c:\Users\금정산2-PC16\Desktop\NS-Project\data\retrain\labels


In [24]:
# ==================================================
# 수동 보강 필요 파일 목록 저장
# 목적: user_character가 자동 탐지되지 않은 이미지 목록 저장
# ==================================================

missing_txt_path = PROJECT_ROOT / "data" / "retrain" / "missing_character_list.txt"

with open(missing_txt_path, "w", encoding="utf-8") as f:
    for txt_name in character_missing:
        img_name = txt_name.replace(".txt", ".jpg")
        f.write(img_name + "\n")

print("저장 완료:", missing_txt_path)

저장 완료: c:\Users\금정산2-PC16\Desktop\NS-Project\data\retrain\missing_character_list.txt


In [25]:
# ==================================================
# 보강 데이터셋 상태 점검
# 목적: retrain/images와 retrain/labels가 1:1로 맞는지 확인
# ==================================================

image_files = sorted(RETRAIN_IMAGES.glob("*.jpg"))
label_files = sorted(RETRAIN_LABELS.glob("*.txt"))

image_stems = {p.stem for p in image_files}
label_stems = {p.stem for p in label_files}

missing_labels = image_stems - label_stems
extra_labels = label_stems - image_stems

print("이미지 수:", len(image_files))
print("라벨 수:", len(label_files))
print("라벨 없는 이미지 수:", len(missing_labels))
print("이미지 없는 라벨 수:", len(extra_labels))

print("라벨 없는 이미지:", missing_labels)
print("이미지 없는 라벨:", extra_labels)

이미지 수: 23
라벨 수: 23
라벨 없는 이미지 수: 0
이미지 없는 라벨 수: 0
라벨 없는 이미지: set()
이미지 없는 라벨: set()
